In [3]:
# cell no - 2
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as models
from torch.utils.data import DataLoader

import numpy as np
import matplotlib.pyplot as plt
import os

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap

# ===== CONFIG =====
MODEL_NAME = "resnet50"  # change only this if needed

BASE_PATH = "/content/drive/MyDrive/4-2/Deep Learning/Assignments/Assignment-2"

CIFAR_PATH = "./data"
FLOWER_TEST_PATH = f"{BASE_PATH}/dataset/flower"

RESULTS_PATH = f"{BASE_PATH}/results/flower_task/{MODEL_NAME}"

BATCH_SIZE = 32
EPOCHS = 5


In [4]:
# cell no - 3
def get_transforms():
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

In [5]:
# cell no - 4
def load_cifar(transform):
    train = datasets.CIFAR10(root=CIFAR_PATH, train=True, download=True, transform=transform)
    test  = datasets.CIFAR10(root=CIFAR_PATH, train=False, download=True, transform=transform)

    train_loader = DataLoader(train, batch_size=BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(test, batch_size=BATCH_SIZE, shuffle=False)

    return train_loader, test_loader

In [6]:
# cell no - 5
def load_flowers(transform):
    dataset = datasets.ImageFolder(FLOWER_TEST_PATH, transform=transform)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)
    return loader, dataset.targets, dataset.classes

In [7]:
# cell no - 6
def get_model(model_name, num_classes=None, pretrained=True):

    if model_name == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        in_features = model.fc.in_features

        if num_classes:
            model.fc = nn.Linear(in_features, num_classes)

        feature_extractor = nn.Sequential(*list(model.children())[:-1])

    else:
        raise ValueError("Only resnet50 implemented for simplicity")

    return model.to(device), feature_extractor.to(device)

In [8]:
# cell no - 7
def extract_features(model, loader):
    model.eval()

    features = []
    labels = []

    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(device)

            out = model(imgs)
            out = out.view(out.size(0), -1)

            features.append(out.cpu().numpy())
            labels.append(lbls.numpy())

    return np.vstack(features), np.hstack(labels)

In [9]:
# cell no - 8
def train_model(model, train_loader):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    model.train()

    for epoch in range(EPOCHS):
        total_loss = 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(imgs)

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

In [10]:
# cell no - 9
def reduce(features):
    results = {}

    results["pca"] = PCA(n_components=2).fit_transform(features)

    n = features.shape[0]
    perplexity = max(2, min(30, n // 3))

    results["tsne"] = TSNE(n_components=2, perplexity=perplexity).fit_transform(features)

    results["umap"] = umap.UMAP(n_components=2).fit_transform(features)

    return results

In [11]:
# cell no - 10
def save_plot(emb, labels, title, path):
    plt.figure(figsize=(6,5))
    plt.scatter(emb[:,0], emb[:,1], c=labels)
    plt.title(title)
    plt.colorbar()
    plt.savefig(path)
    plt.close()

In [13]:
# cell no - 11
def main():

    transform = get_transforms()

    os.makedirs(RESULTS_PATH, exist_ok=True)

    # ===== LOAD DATA =====
    cifar_train, cifar_test = load_cifar(transform)
    flower_loader, flower_labels, flower_classes = load_flowers(transform)

    # ===== BEFORE FINE-TUNING =====
    model, feature_extractor = get_model(MODEL_NAME, pretrained=True)

    flower_features_before, labels = extract_features(feature_extractor, flower_loader)
    reduced_before = reduce(flower_features_before)

    for k, v in reduced_before.items():
        save_plot(v, labels,
                  f"{MODEL_NAME} FLOWERS BEFORE {k}",
                  f"{RESULTS_PATH}/before_{k}.png")

    # ===== AFTER FINE-TUNING ON CIFAR =====
    model_ft, _ = get_model(MODEL_NAME, num_classes=10, pretrained=True)

    train_model(model_ft, cifar_train)

    feature_extractor_ft = nn.Sequential(*list(model_ft.children())[:-1])
    feature_extractor_ft = feature_extractor_ft.to(device)
    feature_extractor_ft.eval()

    flower_features_after, labels = extract_features(feature_extractor_ft, flower_loader)
    reduced_after = reduce(flower_features_after)

    for k, v in reduced_after.items():
        save_plot(v, labels,
                  f"{MODEL_NAME} FLOWERS AFTER {k}",
                  f"{RESULTS_PATH}/after_{k}.png")

    print("✅ DONE - FLOWER TASK COMPLETE")


In [14]:
# cell no - 12
main()

100%|██████████| 170M/170M [00:02<00:00, 67.7MB/s]


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:02<00:00, 44.3MB/s]


Epoch 1, Loss: 526.7171
Epoch 2, Loss: 173.8583
Epoch 3, Loss: 108.4831
Epoch 4, Loss: 81.4835
Epoch 5, Loss: 74.0367
✅ DONE - FLOWER TASK COMPLETE
